## RNN 학습 루프
- 실제로 작성한 RNN 모듈을 가지고 여러 데이터에 대해서 학습 및 평가를 진행할 예정
- RNN의 장단점을 보여주는 것이 목표

### Import Packages

In [ ]:
import random
from typing import Dict
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from vanila_rnn import VanilaRNN

### `nn.Module` 정의
- 학습 대상인 RNN 언어모델 구현을 위해서 직접 구현한 `VanilaRNN` 모듈을 이용
- 입력값으로는 토큰의 임베딩 벡터를 받고, 결과값으로 각 토큰의 예측값을 반환하도록 함

![image.png](https://static.wikidocs.net/images/page/46496/rnnlm1_final_final.PNG)

In [ ]:
class RNNLM(nn.Module):
    def __init__(
        self, vocab_size: int, hidden_size: int, output_size: int, *args, **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.rnn = VanilaRNN(input_size=vocab_size, hidden_size=hidden_size)
        self.linear = nn.Linear(in_features=hidden_size, out_features=output_size)

    def forward(self, x: torch.FloatTensor, hidden = None):
        """
        Args:
            x (batch_size, seq_len, vocab_size): one-hot vector 입력 문자열
            hidden (hidden_size,): RNN의 기본 hidden vector
        """
        if x.ndim == 2:
            x = x.unsqueeze(0)
        output, hidden = self.rnn(x, hidden)
        y = self.linear(output)
        return y, hidden

### 데이터셋 구성
- RNNLM을 이용해서 텍스트의 다음 문자를 바로 예측하는 Task를 수행하고자 함
- 기본 소문자 + 공백 + 기본 문장부호 정도를 Vocabulary로 두고 진행

In [ ]:
def make_vocab(text: str):
    char_set = sorted(set(list(text)))
    chr2idx = {c: i for i, c in enumerate(list(char_set))}
    idx2chr = {i: c for i, c in enumerate(list(char_set))}
    return chr2idx, idx2chr, len(char_set)

chr2idx, idx2chr, vocab_size = make_vocab('abcdefghijklmnopqrstuvwxyz .,?!')

In [ ]:
def encoder(tokens: list, vocab: Dict[str, int]):
    token_ids = [vocab[token] for token in tokens]
    return token_ids

def decoder(token_ids: list, vocab: Dict[int, str]):
    tokens = [vocab[token_id] for token_id in token_ids]
    return tokens
    
print(encoder(list("i love you"), chr2idx))
print(decoder([13, 0, 16, 19, 26, 9, 0, 29, 19, 25], idx2chr))

In [ ]:
def make_repeating_data(pattern: str, length: int):
    text = pattern * (length // len(pattern)) + pattern[:(length % len(pattern))]
    encoded_text = encoder(list(text), chr2idx)
    return encoded_text[:-1], encoded_text[1:]

def make_long_range_data(char: str, N: int, vocab: list):
    vocab_available = vocab[:]
    vocab_available.remove(char)
    padding = [random.choice(vocab_available) for _ in range(N)]

    text = char + ''.join(padding) + char
    encoded_text = encoder(list(text), chr2idx)
    return encoded_text[:-1], encoded_text[1:]

def make_corpus_data(text: str):
    encoded_text = encoder(list(text), chr2idx)
    return encoded_text[:-1], encoded_text[1:]

### 학습 파이프라인 구성

In [ ]:
def train(model: nn.Module, inputs, labels, loss_fn, optimizer, device: str = "cpu"):
    # 1. model을 학습 모드로 설정
    model.train()
    running_loss = 0.0
    
    for X, y in zip(inputs, labels):
        X, y = X.to(device), y.to(device)

        # 2. 이전 step에서의 grad 초기화
        optimizer.zero_grad()

        # 3. Loss 계산 (Forward)
        outputs, hidden = model.forward(X)
        loss = loss_fn(outputs.reshape(-1, vocab_size), y.reshape(-1))

        # 4. Gradient 계산 (Backward)
        loss.backward()

        # 5. Weight 업데이트 (Optimization)
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(inputs)

In [ ]:
# 1. 텍스트 반복 데이터셋 생성
patterns = ["i love you ", "hi", "this is an apple. ", "the weather is hot. ", "wow!"]
repeating_dataset = [make_repeating_data(pattern, 20) for pattern in patterns]
inputs, outputs = zip(*repeating_dataset)
inputs = F.one_hot(torch.LongTensor(inputs), num_classes=vocab_size).float()
outputs = torch.LongTensor(outputs)

In [ ]:
model = RNNLM(vocab_size=vocab_size, hidden_size=128, output_size=vocab_size)
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
num_epochs = 100

for epoch in tqdm(range(num_epochs)):
    train_loss = train(model, inputs, outputs, loss_fn, optimizer)
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f}")

In [ ]:
with torch.no_grad():
    outputs, hidden = model.forward(inputs[0])
    logits = F.softmax(outputs, dim=-1)
    token_ids = logits.argmax(dim=-1)
    decoded = decoder(token_ids.squeeze().tolist(), idx2chr)

In [ ]:
print(''.join(decoder(repeating_dataset[0][0], idx2chr)))
print(''.join(decoded))